In [12]:
import os
import shutil
import fitz 
from PIL import Image
import io

SOURCE_DIR = r"D:\JF_RAG\DataSet 8\VOL00008\IMAGES\0011"
COLOR_PDF_DIR = r"D:\JF_RAG\image"
BW_PDF_DIR = r"D:\JF_RAG\document"

os.makedirs(COLOR_PDF_DIR, exist_ok=True)
os.makedirs(BW_PDF_DIR, exist_ok=True)

def is_image_color(image_bytes, color_tolerance=15, color_threshold=0.02):
    """
    Analyzes an image to determine if it's color or black & white.
    """
    try:
        img = Image.open(io.BytesIO(image_bytes))
        if img.mode in ('L', '1'):
            return False
        img = img.convert('RGB')
        img.thumbnail((200, 200)) 
        
        colors = img.getcolors(maxcolors=100000)
        if colors is None:
            return True
            
        color_pixels = 0
        total_pixels = img.width * img.height
        
        for count, (r, g, b) in colors:
            if max(r, g, b) - min(r, g, b) > color_tolerance:
                color_pixels += count

        if (color_pixels / total_pixels) > color_threshold:
            return True
            
        return False
        
    except Exception as e:
        print(f"Error analyzing image: {e}")
        return False

def sort_pdfs():
    for filename in os.listdir(SOURCE_DIR):
        if not filename.lower().endswith('.pdf'):
            continue
            
        filepath = os.path.join(SOURCE_DIR, filename)
        is_color_pdf = False
        
        try:
            pdf_document = fitz.open(filepath)
            for page_num in range(min(2, len(pdf_document))): 
                page = pdf_document[page_num]
                image_list = page.get_images(full=True)
                
                for img_index, img in enumerate(image_list):
                    xref = img[0]
                    base_image = pdf_document.extract_image(xref)
                    image_bytes = base_image["image"]
                    
                    if is_image_color(image_bytes):
                        is_color_pdf = True
                        break 
                
                if is_color_pdf:
                    break
                    
            pdf_document.close()
            if is_color_pdf:
                print(f"Moving to Color Folder: {filename}")
                shutil.move(filepath, os.path.join(COLOR_PDF_DIR, filename))
            else:
                print(f"Moving to B&W Folder: {filename}")
                shutil.move(filepath, os.path.join(BW_PDF_DIR, filename))
                
        except Exception as e:
            print(f"Failed to process {filename}: {e}")

if __name__ == "__main__":
    print("Starting PDF sorting...")
    sort_pdfs()
    print("Done!")

Starting PDF sorting...
Moving to B&W Folder: EFTA00037903.pdf
Moving to B&W Folder: EFTA00037904.pdf
Moving to B&W Folder: EFTA00037905.pdf
Moving to Color Folder: EFTA00037906.pdf
Moving to B&W Folder: EFTA00037907.pdf
Moving to B&W Folder: EFTA00037908.pdf
Moving to B&W Folder: EFTA00037909.pdf
Moving to B&W Folder: EFTA00037910.pdf
Moving to B&W Folder: EFTA00037911.pdf
Moving to B&W Folder: EFTA00037912.pdf
Moving to B&W Folder: EFTA00037913.pdf
Moving to B&W Folder: EFTA00037914.pdf
Moving to B&W Folder: EFTA00037915.pdf
Moving to B&W Folder: EFTA00037917.pdf
Moving to B&W Folder: EFTA00037918.pdf
Moving to B&W Folder: EFTA00037919.pdf
Moving to B&W Folder: EFTA00037921.pdf
Moving to Color Folder: EFTA00037922.pdf
Moving to B&W Folder: EFTA00037925.pdf
Moving to B&W Folder: EFTA00037926.pdf
Moving to B&W Folder: EFTA00037927.pdf
Moving to B&W Folder: EFTA00037929.pdf
Moving to B&W Folder: EFTA00037930.pdf
Moving to Color Folder: EFTA00037931.pdf
Moving to B&W Folder: EFTA00037932

In [1]:
import os
import shutil
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
import io

IMAGE_DIR = r"D:\JF_RAG\image"
DOC_DIR = r"D:\JF_RAG\document"

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
os.makedirs(DOC_DIR, exist_ok=True)

def contains_significant_text(filepath, text_threshold=50):
    """
    Checks if a PDF contains significant text, either natively or via OCR.
    """
    try:
        pdf_document = fitz.open(filepath)
        for page_num in range(min(2, len(pdf_document))):
            page = pdf_document[page_num]

            native_text = page.get_text().strip()
            if len(native_text) > text_threshold:
                pdf_document.close()
                return True
                
            image_list = page.get_images(full=True)
            for img_index, img in enumerate(image_list):
                xref = img[0]
                base_image = pdf_document.extract_image(xref)
                image_bytes = base_image["image"]

                image = Image.open(io.BytesIO(image_bytes))
                
                ocr_text = pytesseract.image_to_string(image).strip()

                if len(ocr_text) > text_threshold:
                    pdf_document.close()
                    return True
                    
        pdf_document.close()
        return False
        
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return False

def sort_image_pdfs():
    for filename in os.listdir(IMAGE_DIR):
        if not filename.lower().endswith('.pdf'):
            continue
            
        filepath = os.path.join(IMAGE_DIR, filename)
        
        print(f"Analyzing: {filename}...")
        is_document = contains_significant_text(filepath)

        if is_document:
            print(f"--> Contains text! Moving to Document folder.")
            shutil.move(filepath, os.path.join(DOC_DIR, filename))
        else:
            print(f"--> Pure image. Keeping in Image folder.")

if __name__ == "__main__":
    print("Starting PDF OCR sorting...")
    sort_image_pdfs()
    print("Done!")

Starting PDF OCR sorting...
Analyzing: EFTA00000001.pdf...
--> Contains text! Moving to Document folder.
Analyzing: EFTA00000002.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000003.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000004.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000005.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000006.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000007.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000008.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000009.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000010.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000011.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000012.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000013.pdf...
--> Pure image. Keeping in Image folder.
Analyzing: EFTA00000014.pdf...


In [1]:
import os
import shutil
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
import io

DOC_DIR = r"D:\JF_RAG\document"
IMAGE_DIR = r"D:\JF_RAG\image"

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
os.makedirs(IMAGE_DIR, exist_ok=True)

def contains_significant_text(filepath, text_threshold=50):
    """
    Checks if a PDF contains significant text, either natively or via OCR.
    """
    try:
        pdf_document = fitz.open(filepath)
        for page_num in range(min(2, len(pdf_document))):
            page = pdf_document[page_num]
            
            native_text = page.get_text().strip()
            if len(native_text) > text_threshold:
                pdf_document.close()
                return True
                
            image_list = page.get_images(full=True)
            for img_index, img in enumerate(image_list):
                xref = img[0]
                base_image = pdf_document.extract_image(xref)
                image_bytes = base_image["image"]

                image = Image.open(io.BytesIO(image_bytes))
                ocr_text = pytesseract.image_to_string(image).strip()

                if len(ocr_text) > text_threshold:
                    pdf_document.close()
                    return True
                    
        pdf_document.close()
        return False
        
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return False

def move_images_back():
    moved_count = 0
    for filename in os.listdir(DOC_DIR):
        if not filename.lower().endswith('.pdf'):
            continue
            
        filepath = os.path.join(DOC_DIR, filename)
        
        print(f"Checking: {filename}...")
        is_document = contains_significant_text(filepath)
        if not is_document:
            print(f"--> No text found. Moving back to Image folder.")
            shutil.move(filepath, os.path.join(IMAGE_DIR, filename))
            moved_count += 1
        else:
            print(f"--> Text confirmed. Keeping in Document folder.")
            
    print(f"\nCleanup complete! Moved {moved_count} image files back.")

if __name__ == "__main__":
    print("Starting cleanup scan...")
    move_images_back()

Starting cleanup scan...
Checking: EFTA00000001.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000019.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000037.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000046.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000050.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000052.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000068.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000077.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000084.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000094.pdf...
--> No text found. Moving back to Image folder.
Checking: EFTA00000108.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000121.pdf...
--> Text confirmed. Keeping in Document folder.
Checking: EFTA00000131.pdf...
--> Text 

In [2]:
import os
import random
import shutil

DATA_DIR = r"D:\JF_RAG\data"

DOC_DIR = os.path.join(DATA_DIR, "document")
IMAGE_DIR = os.path.join(DATA_DIR, "image")

SAMPLE_DOC_DIR = os.path.join(DATA_DIR, "sample_document")
SAMPLE_IMAGE_DIR = os.path.join(DATA_DIR, "sample_image")

os.makedirs(SAMPLE_DOC_DIR, exist_ok=True)
os.makedirs(SAMPLE_IMAGE_DIR, exist_ok=True)

def create_sample(source_dir, dest_dir, sample_size=100):
    """
    Selects random files from the source and copies them to the destination.
    """
    try:
        files = [f for f in os.listdir(source_dir) if os.path.isfile(os.path.join(source_dir, f))]
        num_to_sample = min(sample_size, len(files))
        selected_files = random.sample(files, num_to_sample)
        for file_name in selected_files:
            src_path = os.path.join(source_dir, file_name)
            dest_path = os.path.join(dest_dir, file_name)
            shutil.copy2(src_path, dest_path)
            
        print(f"Successfully copied {num_to_sample} random files from '{os.path.basename(source_dir)}' to '{os.path.basename(dest_dir)}'.")
        
    except Exception as e:
        print(f"Error processing folder {source_dir}: {e}")

if __name__ == "__main__":
    print("Starting random sampling process...")
    create_sample(DOC_DIR, SAMPLE_DOC_DIR, 100)
    create_sample(IMAGE_DIR, SAMPLE_IMAGE_DIR, 100)
    
    print("Sampling complete! Your CSV files and original directories are unchanged.")

Starting random sampling process...
Successfully copied 100 random files from 'document' to 'sample_document'.
Successfully copied 100 random files from 'image' to 'sample_image'.
Sampling complete! Your CSV files and original directories are unchanged.


In [1]:
import os
import random
import shutil

def move_random_pdfs(src_dir, dest_dir, num_files=500):
    os.makedirs(dest_dir, exist_ok=True)
    pdf_files = [
        f for f in os.listdir(src_dir)
        if f.lower().endswith(".pdf") and os.path.isfile(os.path.join(src_dir, f))
    ]

    if not pdf_files:
        print(f"No PDF files found in {src_dir}")
        return

    selected_files = random.sample(pdf_files, min(num_files, len(pdf_files)))

    for file in selected_files:
        src_path = os.path.join(src_dir, file)
        dest_path = os.path.join(dest_dir, file)
        shutil.move(src_path, dest_path)

    print(f"Moved {len(selected_files)} PDF files from {src_dir} to {dest_dir}")


# ----------- Paths -----------
image_src = r"D:\JF_RAG\image"
image_dest = r"D:\JF_RAG\data\sample_image"

document_src = r"D:\JF_RAG\document"
document_dest = r"D:\JF_RAG\data\sample_document"

# ----------- Execute ----------
move_random_pdfs(image_src, image_dest, 500)
move_random_pdfs(document_src, document_dest, 500)

Moved 500 PDF files from D:\JF_RAG\image to D:\JF_RAG\data\sample_image
Moved 500 PDF files from D:\JF_RAG\document to D:\JF_RAG\data\sample_document
